# import

In [1]:
import sys
sys.path.append('..')
import numpy as np
from StatTools.generators.ndfnoise_generator import ndfnoise
# from StatTools.generators.multi_scale_fractional_generator import MultiScaleFractionalGenerator
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output
import plotly.express as px
from nmot import NMOT

def gen_traj(frame_num: int, ants_num: int, frame_shape: tuple, margin:int = 10,
             hurst_move: float = 0.5, hurst_species: float = 0.5, 
             start_point = None):

    dx = ndfnoise(shape=(frame_num, ants_num), hurst=[hurst_move, hurst_species], normalize=True, dtype=np.float32)
    dy = ndfnoise(shape=(frame_num, ants_num), hurst=[hurst_move, hurst_species], normalize=True, dtype=np.float32)

    x = dx.cumsum(axis=0).round().astype(np.int32)
    y = dy.cumsum(axis=0).round().astype(np.int32)

    if start_point is None:
        start_x = np.random.randint(-margin, frame_shape[1] + margin, (ants_num,))
        start_y = np.random.randint(-margin, frame_shape[0] + margin, (ants_num,))
        start_point = np.stack([start_x, start_y], axis=1)

    trajectories = np.stack([x, y], axis=2)
    trajectories = trajectories + start_point

    return trajectories

In [2]:
class KalmanPredictor():
    def __init__(self):
        self.kf_states = {}

    def _init_kf(self, pos):
        kf = cv.KalmanFilter(dynamParams = 4, measureParams = 2)
        kf.measurementMatrix = np.array([[1,0,0,0],
                                         [0,1,0,0]], np.float32)

        kf.transitionMatrix = np.array([[1,0,1,0],
                                        [0,1,0,1], 
                                        [0,0,1,0], 
                                        [0,0,0,1]], np.float32)

        kf.processNoiseCov = np.eye(4, dtype=np.float32) * 1e-2 
        kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 1e-1
        kf.statePost = np.array([[pos[0]], [pos[1]], [0], [0]], np.float32)

        return kf
    
    def predict_update(self, tracks):
        predictions = []
        for tid in range(tracks.shape[1]):
            kf = self.kf_states[tid]
            x_pred, y_pred, _, _ = kf.predict()
            kf.correct(np.array([[np.float32(tracks[0][tid][0])], [np.float32(tracks[0][tid][1])]], np.float32))
            predictions[tid] = [*x_pred, *y_pred]
            
        return predictions


# few tracks

In [ ]:
np.random.seed(42)
traj = gen_traj(frame_num=1000, 
                ants_num=100, 
                frame_shape=(500, 500), 
                margin = -10,
                hurst_move = 0.5,
                hurst_species = 0.5, 
                start_point = None)


In [ ]:
kp = KalmanPredictor()
kf_dict = {}
for tid in range(traj.shape[1]):
    kf_dict[tid] = kp._init_kf(traj[0, tid])

from collections import defaultdict
predictions = defaultdict(list)
for tid in range(traj.shape[1]):
    
    kf = kf_dict[tid]
    for detections in traj[1:, tid, :]:
        x_pred, y_pred, _, _ = kf.predict()
        kf.correct(np.array([[np.float32(detections[0])], [np.float32(detections[1])]], np.float32))
        predictions[tid].append([*x_pred, *y_pred])
        
kalman_preds = np.stack(list(predictions.values())).transpose((1,0,2))

In [ ]:
fig, ax = plt.subplots(nrows=2)

ax[0].plot(traj[1:,0,0])
ax[0].plot(kalman_preds[:,0,0])
ax[1].plot(traj[1:,0,1])
ax[1].plot(kalman_preds[:,0,1])
ax[0].set_title('2D kalman')
plt.show()

fig, ax = plt.subplots(nrows=2)
ax[0].plot(traj[1:,0,0]-kalman_preds[:,0,0])
ax[1].plot(traj[1:,0,1]-kalman_preds[:,0,1])
ax[0].set_title('err 2D kalman')
plt.show()

rmse = np.sqrt(np.mean((traj[1:]-kalman_preds)**2))
print(f'{rmse=:.4f}')

# rmse vs hurst

In [ ]:
hurst_dict = {}
for hurst in np.arange(0.1, 2.0, 0.1):
    np.random.seed(42)
    traj = gen_traj(frame_num=1000, 
                    ants_num=300, 
                    frame_shape=(500, 500), 
                    margin = -10,
                    hurst_move = hurst,
                    hurst_species = 0.5, 
                    start_point = None)
    kp = KalmanPredictor()
    kf_dict = {}
    for tid in range(traj.shape[1]):
        kf_dict[tid] = kp._init_kf(traj[0, tid])

    from collections import defaultdict
    predictions = defaultdict(list)
    for tid in range(traj.shape[1]):
        
        kf = kf_dict[tid]
        for detections in traj[1:, tid, :]:
            x_pred, y_pred, _, _ = kf.predict()
            kf.correct(np.array([[np.float32(detections[0])], [np.float32(detections[1])]], np.float32))
            predictions[tid].append([*x_pred, *y_pred])
            
    kalman_preds = np.stack(list(predictions.values())).transpose((1,0,2))
    rmse = np.mean(np.sqrt(np.mean((traj[1:]-kalman_preds)**2, axis=0)), axis=1)
    hurst_dict[hurst.round(1)] = rmse

In [ ]:
hurst_df = pd.DataFrame(hurst_dict)

In [ ]:
fig, ax = plt.subplots(nrows=2)
fig.set_size_inches((15,8))
sns.histplot(hurst_df, bins=100, stat='density', kde=True, ax=ax[0], palette='viridis')
sns.violinplot(hurst_df, ax=ax[1], native_scale=True, palette='viridis')
ax[1].plot(hurst_df.median())
ax[0].set_xlabel('RMSE')
ax[1].set_ylabel('RMSE')
ax[1].set_xlabel('H')
ax[1].grid()
plt.show()

# tracker debug

In [6]:
def process_video(
    input_path: str,
    output_path: str = "tracked_output.mp4",
    csv_path: str = "tracks.csv",
    imshow=True,
    roi=None,
    dist2Threshold=50,
    knn_history = 100,
    pred_head = 'kalman'
):
    cap = cv.VideoCapture(input_path)

    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {input_path}")

    fps = cap.get(cv.CAP_PROP_FPS)
    width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
    if output_path is not None:
        fourcc = cv.VideoWriter_fourcc(*"mp4v")
        writer = cv.VideoWriter(output_path, fourcc, fps, (width, height))

    tracker = NMOT(
        warmup_frames=50,
        dist2Threshold=dist2Threshold,
        knn_history=knn_history,
        min_area=8,
        max_area=80,
        max_match_dist=25,
        max_missed=12,
        pred_head=pred_head,
        roi=roi,
    )

    while True:
        ok, frame = cap.read()

        if not ok:
            break

        vis, mask, active_tracks = tracker.update(frame)
        if output_path is not None: 
            writer.write(vis)

        if imshow:
            # cv2.imshow('frame', frame)
            fig, ax = plt.subplots(ncols=2)
            fig.set_size_inches((16, 8))
            
            ax[0].imshow(vis)
            
            

            
            ax[1].imshow(mask)
            
            plt.show()
            
            clear_output(wait=True)
            if cv.waitKey(1) & 0xFF == 27:
                cv.destroyAllWindows()
                if output_path:
                    writer.release()
                break

    cap.release()
    if output_path is not None: 
        writer.release()
    cv.destroyAllWindows()

    df = tracker.save_tracks(csv_path)

    return df

In [11]:
df = process_video(input_path='../data/collision_model_ants.mp4',
                   output_path=None,
                  csv_path='../data/collision_model_ants_lk.csv',
                  dist2Threshold=200,
                  knn_history = 200,
                  imshow=False,
                  pred_head='lucas-kanade')